# B-14: Comprehensive Hyperparameter Tuning for Recursive Rollout

**Objective:** Systematically tune tree models (RF/XGB/LightGBM), SARIMAX, DL models (DLinear/LSTM), and TabPFN for the recursive 365-day daily rollout backtest.

**Strategy:** Manual parameter grid search on 2020-2021 held-out validation fold, then validate winners on full 5-anchor rollout.

**Scope:**
- **B-14a:** Manual grid search for RF/XGB/LightGBM; widened SARIMAX order grid; DL manual parameter loop
- **B-14b:** TabPFN covariate selection (3 variants), hybrid ensemble

In [ ]:
from pathlib import Path
import sys, time, warnings
import numpy as np
import pandas as pd
from itertools import product
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

import models.forecasting_dl as fdl
import models.recursive_rollout as rr
from evaluation.metrics import mase as mase_fn

HOURLY = Path("../../data/Hourly")
RESULTS = Path("../../results")
TOWER = 4
DUM = ["is_t2", "is_t4", "is_t9"]
AR_COLS = ["ar_ch4_dlag1", "ar_ch4_dlag2", "ar_ch4_dlag3", "ar_ch4_dlag7", "ar_ch4_dlag14", "ar_ch4_drm7"]
EXOG_B = ["fx_lsu_dens", "fx_WS_mean", "fx_VPD_mean", "fx_USTAR_mean", "fx_PPFD_mean",
          "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing"]
ANCHOR = pd.Timestamp("2021-12-16")  # Single-anchor smoke test
N_DAYS = 365

print("B-14: Hyperparameter Tuning for Recursive Rollout")
print(f"Anchor: {ANCHOR.date()}")

## Data Loading & Pooled Training

In [ ]:
# Load data
dv = pd.read_csv(HOURLY/"forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
FX_B = [c for c in dv.columns if c.startswith("fx")]
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in [2, 4, 9]}

feat_cols = AR_COLS + FX_B + ["ar_fc_dlag1"] + DUM
target_dates = pd.date_range(ANCHOR + pd.Timedelta(days=1), periods=N_DAYS, freq="D")

# Pooled training data: all rows <= anchor across all towers
pool = []
for t in [2, 4, 9]:
    df = T[t].copy()
    df["target"] = df["y_gapfilled"]
    for d in DUM:
        df[d] = 1.0 if d == f"is_t{t}" else 0.0
    pool.append(df[df.index <= ANCHOR])
tr = pd.concat(pool)
tr = tr[tr["target"].notna()]

print(f"Pooled training data: {len(tr)} rows, {len(feat_cols)} features")

# Add year column for validation fold
tr["year"] = tr.index.year

# Validation fold: 2020-2021 (rows with year in [2020, 2021])
val_idx = tr[tr["year"].isin([2020, 2021])].index
train_idx = tr[~tr["year"].isin([2020, 2021])].index

X_train = tr.loc[train_idx, feat_cols]
y_train = tr.loc[train_idx, "target"]
X_val = tr.loc[val_idx, feat_cols]
y_val = tr.loc[val_idx, "target"]

print(f"Train fold: {len(X_train)} rows")
print(f"Validation fold: {len(X_val)} rows")

## B-14a Part 1: Manual Grid Search for Tree Models

Test bounded parameter grids on 2020-2021 validation fold.

In [ ]:
# Prepare imputed training/validation data
imp = SimpleImputer(strategy="mean")
X_train_imp = imp.fit_transform(X_train)
X_val_imp = imp.transform(X_val)

grid_results = []

# === RANDOM FOREST ===
print("\n=== Random Forest Grid Search ===")
rf_param_combos = list(product(
    [0.3, 0.5, 0.7, 1.0],  # max_features
    [5, 10, 20, 50]  # min_samples_leaf
))
print(f"Testing {len(rf_param_combos)} RF parameter combinations...")

for max_feat, min_leaf in rf_param_combos:
    rf = RandomForestRegressor(n_estimators=500, max_features=max_feat, 
                               min_samples_leaf=min_leaf, n_jobs=-1, random_state=42)
    rf.fit(X_train_imp, y_train)
    y_pred = rf.predict(X_val_imp)
    r2 = r2_score(y_val, y_pred)
    grid_results.append({
        "model": "RF", "max_features": max_feat, "min_samples_leaf": min_leaf,
        "val_r2": r2
    })

rf_best = max(grid_results, key=lambda x: x["val_r2"])
print(f"Best RF: max_features={rf_best['max_features']}, min_samples_leaf={rf_best['min_samples_leaf']}, R²={rf_best['val_r2']:.4f}")

# === XGBOOST ===
print("\n=== XGBoost Grid Search ===")
xgb_param_combos = list(product(
    [2, 3, 4, 6],  # max_depth
    [0.01, 0.02, 0.05],  # learning_rate
    [1, 5, 10]  # min_child_weight
))
print(f"Testing {len(xgb_param_combos)} XGB parameter combinations...")

for max_depth, lr, min_child in xgb_param_combos:
    xgb = XGBRegressor(max_depth=max_depth, learning_rate=lr, min_child_weight=min_child,
                      n_estimators=400, subsample=0.8, colsample_bytree=0.8,
                      n_jobs=-1, random_state=42)
    xgb.fit(X_train_imp, y_train)
    y_pred = xgb.predict(X_val_imp)
    r2 = r2_score(y_val, y_pred)
    grid_results.append({
        "model": "XGB", "max_depth": max_depth, "learning_rate": lr,
        "min_child_weight": min_child, "val_r2": r2
    })

xgb_best = max([r for r in grid_results if r["model"] == "XGB"], key=lambda x: x["val_r2"])
print(f"Best XGB: max_depth={xgb_best['max_depth']}, lr={xgb_best['learning_rate']}, min_child_weight={xgb_best['min_child_weight']}, R²={xgb_best['val_r2']:.4f}")

# === LIGHTGBM ===
print("\n=== LightGBM Grid Search ===")
lgb_param_combos = list(product(
    [7, 15, 31, 63],  # num_leaves
    [10, 20, 50],  # min_child_samples
    [0.01, 0.02, 0.05]  # learning_rate
))
print(f"Testing {len(lgb_param_combos)} LGB parameter combinations...")

for num_leaves, min_child, lr in lgb_param_combos:
    lgb = LGBMRegressor(num_leaves=num_leaves, min_child_samples=min_child,
                       learning_rate=lr, n_estimators=400, subsample=0.8,
                       colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=-1)
    lgb.fit(X_train_imp, y_train)
    y_pred = lgb.predict(X_val_imp)
    r2 = r2_score(y_val, y_pred)
    grid_results.append({
        "model": "LGB", "num_leaves": num_leaves, "min_child_samples": min_child,
        "learning_rate": lr, "val_r2": r2
    })

lgb_best = max([r for r in grid_results if r["model"] == "LGB"], key=lambda x: x["val_r2"])
print(f"Best LGB: num_leaves={lgb_best['num_leaves']}, min_child_samples={lgb_best['min_child_samples']}, lr={lgb_best['learning_rate']}, R²={lgb_best['val_r2']:.4f}")

# Save full grid results
pd.DataFrame(grid_results).to_csv(RESULTS/"b14_tree_grid_search.csv", index=False)
print(f"\nGrid search results saved to b14_tree_grid_search.csv")

## B-14a Part 2: SARIMAX Widened Order Grid

In [ ]:
print("\n=== SARIMAX Order Grid Search (AIC) ===")
df4 = T[TOWER]
y = df4["y_gapfilled"].astype(float)
X = df4[EXOG_B].astype(float).ffill().bfill()
y_tr, X_tr = y.loc[:ANCHOR], X.loc[:ANCHOR]

sarimax_results = []
best_aic = None
best_order = None

for p in [1, 2, 3]:
    for q in [0, 1, 2]:
        try:
            m = SARIMAX(y_tr, exog=X_tr, order=(p, 1, q), enforce_stationarity=False, enforce_invertibility=False)
            res = m.fit(disp=False, maxiter=50)
            sarimax_results.append({"p": p, "q": q, "AIC": res.aic})
            if best_aic is None or res.aic < best_aic:
                best_aic = res.aic
                best_order = (p, 1, q)
        except:
            pass

pd.DataFrame(sarimax_results).sort_values("AIC").to_csv(RESULTS/"b14_sarimax_grid.csv", index=False)
print(f"Best SARIMAX order: {best_order}, AIC={best_aic:.2f}")

## B-14a Part 3: DLinear/LSTM Parameter Grid Loop

In [ ]:
print("\n=== DL Parameter Grid Search ===")
device = fdl.get_device()
print(f"Device: {device}")

track = "B"
m = fdl.load_matrix(HOURLY/"forecast_features_v2.csv")

# 2-fold split: train ≤ 2021-06-30, validate 2021-07-01 to 2021-12-31
cutoff_train = pd.Timestamp("2021-06-30 23:59")
cutoff_val = pd.Timestamp("2021-12-31 23:59")

W = fdl.build_windows(m, track)
train_parts_dl, val_parts_dl = [], []
for t in [2, 4, 9]:
    ttime = pd.DatetimeIndex(W[t]["ttime"][:, -1])
    train_parts_dl.append(fdl._subset(W[t], ttime <= cutoff_train))
    val_parts_dl.append(fdl._subset(W[t], (ttime > cutoff_train) & (ttime <= cutoff_val)))

train_dl = fdl._cat(train_parts_dl)
val_dl = fdl._cat(val_parts_dl)
print(f"DL train windows: {len(train_dl['enc'])}, val windows: {len(val_dl['enc'])}")

se_dl, sd_dl = fdl.Scaler().fit(train_dl["enc"]), fdl.Scaler().fit(train_dl["dec"])
yv = train_dl["y"][np.isfinite(train_dl["y"])]
mu_dl, sdy_dl = float(yv.mean()), float(yv.std() + 1e-6)
train_dl["enc"], train_dl["dec"] = se_dl.tf(train_dl["enc"]), sd_dl.tf(train_dl["dec"])
val_dl["enc"], val_dl["dec"] = se_dl.tf(val_dl["enc"]), sd_dl.tf(val_dl["dec"])
n_enc, n_dec = train_dl["enc"].shape[-1], train_dl["dec"].shape[-1]

dl_grid_results = []
for model_name in ["DLinear", "LSTM"]:
    print(f"\n{model_name} parameter grid:")
    for hidden in [16, 32, 64]:
        for lr in [1e-3, 5e-4]:
            for wd in [0, 1e-3]:
                model_dl = fdl.build_model(model_name, 28, 14, n_enc, n_dec, 3)
                fdl.train_model(model_dl, train_dl, device, epochs=20, ch4_mu=mu_dl, ch4_sd=sdy_dl,
                               seed=0, weight_decay=wd, val_data=val_dl, patience=5)
                val_pred = fdl.predict(model_dl, val_dl, device, mu_dl, sdy_dl)
                val_loss = float(np.nanmean((val_pred - fdl._unscale_y(val_dl["dec"], sd_dl, mu_dl))**2))
                dl_grid_results.append({"model": model_name, "hidden": hidden, "lr": lr, "wd": wd,
                                       "val_loss": val_loss})
                print(f"  h={hidden}, lr={lr:.1e}, wd={wd:.1e}: loss={val_loss:.4f}")

pd.DataFrame(dl_grid_results).to_csv(RESULTS/"b14_dl_grid.csv", index=False)

for model_name in ["DLinear", "LSTM"]:
    best = min([r for r in dl_grid_results if r["model"] == model_name], key=lambda x: x["val_loss"])
    print(f"Best {model_name}: h={best['hidden']}, lr={best['lr']:.1e}, wd={best['wd']:.1e}, loss={best['val_loss']:.4f}")

## B-14b: TabPFN Covariate Selection

In [ ]:
print("\n=== TabPFN Covariate Selection ===")
import os

if os.environ.get("TABPFN_TOKEN"):
    print("TabPFN available (TABPFN_TOKEN set)")
    print("Will test 3 covariate variants in multi-anchor validation")
else:
    print("TabPFN not available (TABPFN_TOKEN not set) - will skip in multi-anchor")

## Summary: Winning Hyperparameters

In [ ]:
print("\n" + "="*70)
print("=== WINNING HYPERPARAMETERS (from validation fold) ===")
print("="*70)

print(f"\nRF: max_features={rf_best['max_features']}, min_samples_leaf={rf_best['min_samples_leaf']}")
print(f"XGB: max_depth={xgb_best['max_depth']}, lr={xgb_best['learning_rate']}, min_child_weight={xgb_best['min_child_weight']}")
print(f"LGB: num_leaves={lgb_best['num_leaves']}, min_child_samples={lgb_best['min_child_samples']}, lr={lgb_best['learning_rate']}")
print(f"SARIMAX: order={best_order}")

print(f"\nDL models: see grid results in b14_dl_grid.csv")
print(f"\nAll results saved. Ready for multi-anchor validation via b14_multi_anchor.py")